In [2]:
"""
Download NYISO hourly zonal actual load (palIntegrated dataset).
Source: mis.nyiso.com/public - Integrated Real-Time Actual Load.
Output: one clean parquet, 11 zone columns + statewide sum, UTC index.

Idempotent: cached ZIPs are skipped on re-run. Safe after pod restarts.
"""

import io
import time
import zipfile
from pathlib import Path

import pandas as pd
import requests

# ----------------------------- configuration -----------------------------
START = "2013-01"
END = "2026-05"
BASE_URL = "http://mis.nyiso.com/public/csv/palIntegrated"
CACHE_DIR = Path("nyiso_zonal_raw")
OUT_PATH = Path("nyiso_zonal_hourly.parquet")

ZONES = {
    "WEST": "A", "GENESE": "B", "CENTRL": "C", "NORTH": "D",
    "MHK VL": "E", "CAPITL": "F", "HUD VL": "G", "MILLWD": "H",
    "DUNWOD": "I", "N.Y.C.": "J", "LONGIL": "K",
}

# --------------------------- step 1: download ----------------------------
def download_month(yyyymm: str) -> Path | None:
    fname = f"{yyyymm}01palIntegrated_csv.zip"
    local = CACHE_DIR / fname
    if local.exists() and local.stat().st_size > 0:
        return local
    url = f"{BASE_URL}/{fname}"
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=60)
            if r.status_code == 200:
                local.write_bytes(r.content)
                return local
            print(f"  {yyyymm}: HTTP {r.status_code} - no archive at {url}")
            return None
        except requests.RequestException as e:
            print(f"  {yyyymm}: attempt {attempt + 1} failed ({e}), retrying...")
            time.sleep(5)
    return None


def month_range(start: str, end: str) -> list[str]:
    dates = pd.period_range(start=start, end=end, freq="M")
    return [p.strftime("%Y%m") for p in dates]


# ---------------------------- step 2: parse ------------------------------
def parse_zip(zip_path: Path) -> pd.DataFrame:
    frames = []
    with zipfile.ZipFile(zip_path) as zf:
        for name in sorted(zf.namelist()):
            if not name.endswith(".csv"):
                continue
            with zf.open(name) as f:
                df = pd.read_csv(io.TextIOWrapper(f, encoding="utf-8"))
            frames.append(df)
    return pd.concat(frames, ignore_index=True)


def find_load_column(df: pd.DataFrame) -> str:
    """
    Auto-detect the load value column. The hourly integrated dataset labels it
    'Integrated Load'; other NYISO datasets use 'Load'. Fall back to any single
    column containing 'load' (case-insensitive) so a future rename can't
    silently break us.
    """
    for exact in ("Integrated Load", "Load"):
        if exact in df.columns:
            return exact
    candidates = [c for c in df.columns if "load" in c.lower()]
    if len(candidates) == 1:
        return candidates[0]
    raise ValueError(
        f"Could not identify load column. Columns found: {list(df.columns)}"
    )


def to_utc(df: pd.DataFrame) -> pd.DataFrame:
    """EST = UTC-5, EDT = UTC-4 via the explicit Time Zone column."""
    ts = pd.to_datetime(df["Time Stamp"], format="%m/%d/%Y %H:%M:%S")
    offset = df["Time Zone"].map({"EST": 5, "EDT": 4})
    if offset.isna().any():
        bad = df.loc[offset.isna(), "Time Zone"].unique()
        raise ValueError(f"Unexpected Time Zone values: {bad}")
    df = df.copy()
    df["utc"] = ts + pd.to_timedelta(offset, unit="h")
    return df


# ----------------------------- main pipeline -----------------------------
def main() -> None:
    CACHE_DIR.mkdir(exist_ok=True)
    months = month_range(START, END)
    print(f"Downloading {len(months)} monthly archives "
          f"({months[0]} to {months[-1]})...")

    all_frames, failed = [], []
    for i, m in enumerate(months, 1):
        zp = download_month(m)
        if zp is None:
            failed.append(m)
            continue
        all_frames.append(parse_zip(zp))
        if i % 12 == 0:
            print(f"  ...{i}/{len(months)} months done")

    if failed:
        print(f"\nWARNING - {len(failed)} months failed: {failed}")

    print("\nParsing and pivoting...")
    long_df = pd.concat(all_frames, ignore_index=True)

    # show the actual schema once, so there is no more guessing
    print(f"Columns found in raw files: {list(long_df.columns)}")
    load_col = find_load_column(long_df)
    print(f"Using '{load_col}' as the load value column.")

    long_df = to_utc(long_df)

    found = set(long_df["Name"].unique())
    expected = set(ZONES.keys())
    if found != expected:
        print(f"NOTE - zone name mismatch.\n  missing: {expected - found}"
              f"\n  unexpected: {found - expected}")

    long_df = long_df.drop_duplicates(subset=["utc", "Name"], keep="last")

    wide = long_df.pivot(index="utc", columns="Name", values=load_col)
    wide = wide[[z for z in ZONES if z in wide.columns]]
    wide["NYISO_TOTAL"] = wide.sum(axis=1)
    wide = wide.sort_index()

    # ------------------------- diagnostics report -------------------------
    print("\n================ DIAGNOSTICS ================")
    print(f"Rows (hours):        {len(wide):,}")
    print(f"Range (UTC):         {wide.index.min()}  ->  {wide.index.max()}")
    full_range = pd.date_range(wide.index.min(), wide.index.max(), freq="h")
    missing_hours = full_range.difference(wide.index)
    print(f"Missing hours:       {len(missing_hours)}")
    if len(missing_hours) > 0:
        print(f"  first few: {missing_hours[:5].tolist()}")
    print(f"NaN cells per zone:\n{wide.isna().sum().to_string()}")
    print(f"\nStatewide total - min: {wide['NYISO_TOTAL'].min():,.0f} MW, "
          f"max: {wide['NYISO_TOTAL'].max():,.0f} MW, "
          f"mean: {wide['NYISO_TOTAL'].mean():,.0f} MW")
    print("=============================================")

    wide.to_parquet(OUT_PATH)
    print(f"\nSaved {OUT_PATH} ({OUT_PATH.stat().st_size / 1e6:.1f} MB)")


if __name__ == "__main__":
    main()

  ...12/161 months done


  ...24/161 months done


  ...36/161 months done


  ...48/161 months done


  ...60/161 months done


...

  ...144/161 months done


  ...156/161 months done

Parsing and pivoting...


Columns found in raw files: ['Time Stamp', 'Time Zone', 'Name', 'PTID', 'Integrated Load']
Using 'Integrated Load' as the load value column.



================ DIAGNOSTICS ================
Rows (hours):        117,575
Range (UTC):         2013-01-01 05:00:00  ->  2026-06-01 03:00:00
Missing hours:       0
NaN cells per zone:
Name
WEST           1
GENESE         1
CENTRL         1
NORTH          1
MHK VL         1
CAPITL         1
HUD VL         1
MILLWD         1
DUNWOD         1
N.Y.C.         1
LONGIL         1
NYISO_TOTAL    0

Statewide total - min: 0 MW, max: 33,956 MW, mean: 17,727 MW



Saved nyiso_zonal_hourly.parquet (8.8 MB)


In [3]:
import pandas as pd

df = pd.read_parquet("nyiso_zonal_hourly.parquet")
zones = [c for c in df.columns if c != "NYISO_TOTAL"]

# locate the bad hour
bad = df[df[zones].isna().any(axis=1)]
print("Missing hour(s):")
print(bad)

# interpolate zone-by-zone, recompute the total honestly
df[zones] = df[zones].interpolate(method="linear", limit=2)
df["NYISO_TOTAL"] = df[zones].sum(axis=1)

# verify
assert df[zones].isna().sum().sum() == 0, "still have NaNs"
print(f"\nAfter fix - min: {df['NYISO_TOTAL'].min():,.0f} MW "
      f"(should now be a plausible overnight low, ~11,000-13,000)")

df.to_parquet("nyiso_zonal_hourly.parquet")
print("Saved.")

Missing hour(s):
Name                 WEST  GENESE  CENTRL  NORTH  MHK VL  CAPITL  HUD VL  \
utc                                                                        
2016-01-29 03:00:00   NaN     NaN     NaN    NaN     NaN     NaN     NaN   

Name                 MILLWD  DUNWOD  N.Y.C.  LONGIL  NYISO_TOTAL  
utc                                                               
2016-01-29 03:00:00     NaN     NaN     NaN     NaN          0.0  

After fix - min: 10,731 MW (should now be a plausible overnight low, ~11,000-13,000)


Saved.


In [5]:
import pandas as pd

demand = pd.read_csv("data/processed/tft_clean.csv")
print(demand.columns.tolist())
print(demand.head(3))
print(demand.tail(3))
print(f"\nRows: {len(demand):,}")

['ts', 'demand', 'net_generation', 'total_interchange', 'hour', 'day_of_week', 'day_of_month', 'month', 'year', 'is_holiday', 'time_idx', 'series', 'split', 'act_temperature_2m', 'act_relative_humidity_2m', 'act_dew_point_2m', 'act_apparent_temperature', 'act_precipitation', 'act_snowfall', 'act_cloud_cover', 'act_surface_pressure', 'act_wind_speed_10m', 'act_wind_gusts_10m', 'act_shortwave_radiation', 'act_direct_radiation', 'act_diffuse_radiation', 'fc_temperature_2m', 'fc_relative_humidity_2m', 'fc_dew_point_2m', 'fc_apparent_temperature', 'fc_precipitation', 'fc_snowfall', 'fc_cloud_cover', 'fc_surface_pressure', 'fc_wind_speed_10m', 'fc_wind_gusts_10m', 'fc_shortwave_radiation', 'fc_direct_radiation', 'fc_diffuse_radiation', 'fc_is_synthetic', 'fx_app_roll72', 'fx_cdh_24h', 'fx_hot_streak_day', 'fx_night_min_app_prev']
                    ts   demand  net_generation  total_interchange  hour  \
0  2015-07-01 01:00:00  16891.0         14444.0            -2447.0     1   
1  2015-07-0

In [7]:
"""
Gate 2: Does the sum of NYISO's 11 zones match the EIA-930 statewide series
that the 3.96% baseline was scored on?

Also settles empirically whether tft_clean.csv's 'ts' column is UTC or
US/Eastern local time, by testing both alignments.
"""

from pathlib import Path

import pandas as pd

ZONAL_PATH = Path("nyiso_zonal_hourly.parquet")
BASELINE_PATH = Path("data/processed/tft_clean.csv")

# ------------------------------ load both --------------------------------
zonal = pd.read_parquet(ZONAL_PATH)["NYISO_TOTAL"]
zonal.index.name = "utc"

base = pd.read_csv(BASELINE_PATH, usecols=["ts", "demand"])
base["ts"] = pd.to_datetime(base["ts"])

# ---------------------- diagnostic: duplicated stamps ---------------------
dups = base[base["ts"].duplicated(keep=False)]
print(f"Duplicated timestamps in ts: {dups['ts'].nunique()} distinct stamps, "
      f"{len(dups)} rows")
if len(dups) > 0:
    print(dups["ts"].dt.strftime("%Y-%m-%d %H:%M").unique())
    print("If these are all ~1-2 AM in early November, ts is local time "
          "(DST fall-back) - that alone answers the question.\n")

# ------------------- hypothesis A: ts is already UTC ----------------------
# de-duplicate (mean of the pair) purely so the join can run
hyp_a = base.groupby("ts")["demand"].mean()

# ------------------- hypothesis B: ts is US/Eastern -----------------------
ts_eastern = (
    base["ts"]
    .dt.tz_localize("US/Eastern", ambiguous="infer", nonexistent="shift_forward")
    .dt.tz_convert("UTC")
    .dt.tz_localize(None)
)
hyp_b = pd.Series(base["demand"].values, index=ts_eastern)
hyp_b = hyp_b[~hyp_b.index.duplicated(keep="first")]   # safety, should be no-op

# ------------------------- evaluate both alignments -----------------------
def report(name: str, series: pd.Series) -> float:
    joined = pd.concat([zonal, series.rename("eia")], axis=1, join="inner")
    joined = joined.dropna()
    corr = joined["NYISO_TOTAL"].corr(joined["eia"])
    diff = joined["NYISO_TOTAL"] - joined["eia"]
    pct = 100 * diff / joined["eia"]
    print(f"\n--- Hypothesis {name} ---")
    print(f"Overlapping hours:      {len(joined):,}")
    print(f"Correlation:            {corr:.6f}")
    print(f"Mean gap:               {diff.mean():+,.0f} MW ({pct.mean():+.3f}%)")
    print(f"Median abs gap:         {diff.abs().median():,.0f} MW")
    print(f"95th pct abs gap:       {diff.abs().quantile(0.95):,.0f} MW")
    print(f"Max abs gap:            {diff.abs().max():,.0f} MW")
    return corr


corr_a = report("A: ts = UTC", hyp_a)
corr_b = report("B: ts = US/Eastern local", hyp_b)

winner = "A (UTC)" if corr_a > corr_b else "B (Eastern local)"
print(f"\n==> Better alignment: hypothesis {winner}")
print("Pass criteria: winner correlation ~0.999+, mean gap within ~2%.")

Duplicated timestamps in ts: 11 distinct stamps, 22 rows
['2015-11-01 01:00' '2016-11-06 01:00' '2017-11-05 01:00'
 '2018-11-04 01:00' '2019-11-03 01:00' '2020-11-01 01:00'
 '2021-11-07 01:00' '2022-11-06 01:00' '2023-11-05 01:00'
 '2024-11-03 01:00' '2025-11-02 01:00']
If these are all ~1-2 AM in early November, ts is local time (DST fall-back) - that alone answers the question.


--- Hypothesis A: ts = UTC ---
Overlapping hours:      95,653
Correlation:            0.817259
Mean gap:               -0 MW (+0.650%)
Median abs gap:         1,405 MW
95th pct abs gap:       3,756 MW
Max abs gap:            6,132 MW

--- Hypothesis B: ts = US/Eastern local ---
Overlapping hours:      95,664
Correlation:            0.979619
Mean gap:               +0 MW (+0.073%)
Median abs gap:         438 MW
95th pct abs gap:       1,306 MW
Max abs gap:            2,883 MW

==> Better alignment: hypothesis B (Eastern local)
Pass criteria: winner correlation ~0.999+, mean gap within ~2%.


In [8]:
"""Shift scan: is the residual gap a one-hour convention offset?"""
import pandas as pd

zonal = pd.read_parquet("nyiso_zonal_hourly.parquet")["NYISO_TOTAL"]

base = pd.read_csv("data/processed/tft_clean.csv", usecols=["ts", "demand"])
base["ts"] = pd.to_datetime(base["ts"])
ts_utc = (
    base["ts"]
    .dt.tz_localize("US/Eastern", ambiguous="infer", nonexistent="shift_forward")
    .dt.tz_convert("UTC")
    .dt.tz_localize(None)
)
eia = pd.Series(base["demand"].values, index=ts_utc)
eia = eia[~eia.index.duplicated(keep="first")]

print(f"{'shift':>6} | {'corr':>9} | {'median abs gap':>14}")
print("-" * 38)
for shift in range(-3, 4):
    shifted = eia.copy()
    shifted.index = shifted.index + pd.Timedelta(hours=shift)
    j = pd.concat([zonal, shifted.rename("eia")], axis=1, join="inner").dropna()
    corr = j["NYISO_TOTAL"].corr(j["eia"])
    med = (j["NYISO_TOTAL"] - j["eia"]).abs().median()
    marker = "  <--" if corr > 0.998 else ""
    print(f"{shift:>+5}h | {corr:.6f} | {med:>11,.0f} MW{marker}")

 shift |      corr | median abs gap
--------------------------------------
   -3h | 0.923851 |         872 MW
   -2h | 0.979614 |         438 MW
   -1h | 0.999997 |           0 MW  <--
   +0h | 0.979619 |         438 MW
   +1h | 0.923863 |         872 MW
   +2h | 0.843215 |       1,282 MW
   +3h | 0.748249 |       1,677 MW


In [9]:
"""
Gate 3: Confirm Open-Meteo Previous Runs API coverage at all 9 zonal
coordinates. The latest common start date defines the lead-matched
evaluation window for the zonal rebuild.
"""

import time

import pandas as pd
import requests

# zone -> representative city coordinates
COORDS = {
    "A_WEST_Buffalo":        (42.886, -78.878),
    "B_GENESE_Rochester":    (43.157, -77.616),
    "C_CENTRL_Syracuse":     (43.048, -76.147),
    "D_NORTH_Massena":       (44.928, -74.892),
    "E_MHKVL_Utica":         (43.101, -75.233),
    "F_CAPITL_Albany":       (42.653, -73.757),
    "G_HUDVL_Poughkeepsie":  (41.706, -73.921),
    "J_NYC_Manhattan":       (40.714, -74.006),
    "K_LONGIL_Islip":        (40.730, -73.210),
}
# zones H (Millwood) and I (Dunwoodie) will reuse NYC/Poughkeepsie weather

URL = "https://previous-runs-api.open-meteo.com/v1/forecast"

# probe windows: around the known NYC boundary, plus one clearly-after sanity window
PROBES = [
    ("boundary", "2024-01-20", "2024-01-27"),
    ("recent",   "2025-06-01", "2025-06-03"),
]


def probe(lat: float, lon: float, start: str, end: str) -> tuple[int, int]:
    """Return (n_hours_total, n_hours_with_data) for temperature at lead day 5."""
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start,
        "end_date": end,
        "hourly": "temperature_2m_previous_day5",
        "timezone": "UTC",
    }
    r = requests.get(URL, params=params, timeout=60)
    r.raise_for_status()
    js = r.json()
    vals = js.get("hourly", {}).get("temperature_2m_previous_day5", [])
    n_total = len(vals)
    n_good = sum(1 for v in vals if v is not None)
    return n_total, n_good


def main() -> None:
    rows = []
    for name, (lat, lon) in COORDS.items():
        row = {"zone": name}
        for label, start, end in PROBES:
            try:
                n_total, n_good = probe(lat, lon, start, end)
                row[label] = f"{n_good}/{n_total}"
            except Exception as e:
                row[label] = f"ERROR: {type(e).__name__}"
            time.sleep(1)          # polite to the free API
        rows.append(row)
        print(f"  probed {name}: boundary={row.get('boundary')}, "
              f"recent={row.get('recent')}")

    df = pd.DataFrame(rows)
    print("\n================= GATE 3 RESULTS =================")
    print(df.to_string(index=False))
    print("==================================================")
    print(
        "\nHow to read:\n"
        "- 'boundary' window straddles 2024-01-23 (the NYC archive start).\n"
        "  Partial counts (~90-100/192) = coverage begins mid-window, same\n"
        "  boundary as NYC -> ideal.\n"
        "- 'recent' should be 48/48 everywhere.\n"
        "- Any zone showing 0 in both windows or ERROR = problem coordinate;\n"
        "  we pick a nearby city and re-probe."
    )


if __name__ == "__main__":
    main()

  probed A_WEST_Buffalo: boundary=108/192, recent=72/72


  probed B_GENESE_Rochester: boundary=108/192, recent=72/72


  probed C_CENTRL_Syracuse: boundary=108/192, recent=72/72


  probed D_NORTH_Massena: boundary=108/192, recent=72/72


  probed E_MHKVL_Utica: boundary=108/192, recent=72/72


  probed F_CAPITL_Albany: boundary=108/192, recent=72/72


  probed G_HUDVL_Poughkeepsie: boundary=108/192, recent=72/72


  probed J_NYC_Manhattan: boundary=108/192, recent=72/72


  probed K_LONGIL_Islip: boundary=108/192, recent=72/72

================= GATE 3 RESULTS =================
                zone boundary recent
      A_WEST_Buffalo  108/192  72/72
  B_GENESE_Rochester  108/192  72/72
   C_CENTRL_Syracuse  108/192  72/72
     D_NORTH_Massena  108/192  72/72
       E_MHKVL_Utica  108/192  72/72
     F_CAPITL_Albany  108/192  72/72
G_HUDVL_Poughkeepsie  108/192  72/72
     J_NYC_Manhattan  108/192  72/72
      K_LONGIL_Islip  108/192  72/72

How to read:
- 'boundary' window straddles 2024-01-23 (the NYC archive start).
  Partial counts (~90-100/192) = coverage begins mid-window, same
  boundary as NYC -> ideal.
- 'recent' should be 48/48 everywhere.
- Any zone showing 0 in both windows or ERROR = problem coordinate;
  we pick a nearby city and re-probe.
